In [0]:
%sql
USE hive_metastore.rxhman;

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
customer_df = spark.read.table("customers")

In [0]:
customers_df = customer_df 

del customer_df

In [0]:
display(customers_df)

In [0]:
display(customers_df.select("customer_id", "email"))

In [0]:
from pyspark.sql.functions import col

display( customers_df.select(col("customer_id"), col("email").alias("customer_email"), col("updated").cast("timestamp")) )

In [0]:
from pyspark.sql.functions import col, split

display(customers_df.withColumn("domain", split(col("email"), "@").getItem(1)))

In [0]:
display(customers_df.withColumnRenamed("email", "customer_email"))

In [0]:
new_customers_df = customers_df.withColumnRenamed("email", "customer_email")
display(new_customers_df)

In [0]:
from pyspark.sql.functions import col, split

new_customers_df = customers_df.withColumn("customer_domain", split(col("email"), "@").getItem(1)) \
                               .withColumnRenamed("email", "customer_email") \
                               .drop("profile", "updated")
 
display(new_customers_df)

In [0]:
orders_df = spark.table("orders")

display(orders_df.describe())

In [0]:
display(orders_df.summary())

In [0]:
deduped_df = orders_df.dropDuplicates()
display(deduped_df)

In [0]:
deduped_df = orders_df.dropDuplicates(["customer_id"])
display(deduped_df)

In [0]:
display(customers_df.summary())

In [0]:
display(customers_df)

In [0]:
clean_df = customers_df.dropna()
display(clean_df)

In [0]:
joined_df = customers_df.join(orders_df, "customer_id", "left")
display(joined_df)

In [0]:
anti_joined_df = customers_df.join(orders_df, "customer_id", "left_anti")
display(anti_joined_df)

In [0]:
display(orders_df.where("customer_id = 'C01194'"))

In [0]:
books_df = spark.table("books")

cross_joined_df = customers_df.crossJoin(books_df)
display(cross_joined_df.orderBy("customer_id"))

In [0]:
print(f"customers count: {customers_df.count()}")
print(f"books count: {books_df.count()}")
print(f"customer-book pair: {cross_joined_df.count()}")

In [0]:
from pyspark.sql.functions import explode, broadcast

exploded_orders_df = (orders_df.withColumn("book", explode("books"))
                                .select("*", "book.book_id", "book.subtotal")
                                .drop("books", "book")
                     )

orders_details_df = exploded_orders_df.join(broadcast(books_df), "book_id", "inner")
display(orders_details_df)

In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 20 * 1024 * 1024)


In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [0]:
customers_details_df = spark.sql("""SELECT customer_id, email, updated,
                                        profile:first_name, profile:last_name, profile:gender,
                                        profile:address:street,
                                        profile:address:city,
                                        profile:address:country
                                    FROM customers
                                """)

display(customers_details_df)

In [0]:
customers_details_df.write.mode("overwrite").saveAsTable("customers_details")

In [0]:
%sql
SELECT * FROM customers_details;